# Q20: Fine-tuning a Foundation Model
**Topic:** LLM | **Difficulty:** Medium | **Time:** 25 min
**Sheet:** GrindLLM50

---

## Question
How do you fine-tune a foundation model for a specific downstream task?

## Detailed Answer

### End-to-End Pipeline

**1. Task Definition & Data Preparation**
- Define input/output format (e.g., instruction → response)
- Collect/annotate dataset (min 100-1000 examples for LoRA)
- Train/val/test split
- Format as instruction templates:
```
<|system|>You are a helpful medical assistant.
<|user|>{question}
<|assistant|>{answer}
```

**2. Choose Fine-tuning Strategy**
- **Full fine-tuning**: All parameters (need lots of data + compute)
- **LoRA**: Low-rank adapters on attention layers (recommended default)
- **QLoRA**: 4-bit quantized base + LoRA (memory constrained)

**3. Hyperparameters**
| Parameter | Typical Value |
|-----------|---------------|
| Learning rate | 1e-5 to 2e-4 |
| Batch size | 4-32 (with gradient accumulation) |
| Epochs | 1-5 |
| LoRA rank | 8-64 |
| LoRA alpha | 16-128 |
| LoRA target | q_proj, v_proj (or all linear) |
| Weight decay | 0.01-0.1 |
| Warmup | 5-10% of steps |

**4. Training with HuggingFace**
```python
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

model = AutoModelForCausalLM.from_pretrained('model_id', load_in_4bit=True)
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','v_proj'])
model = get_peft_model(model, lora_config)

trainer = SFTTrainer(model=model, train_dataset=dataset, ...)
trainer.train()
```

**5. Evaluation**
- Task-specific metrics (F1, BLEU, ROUGE, accuracy)
- Perplexity on validation set
- Human evaluation for open-ended tasks
- Check for catastrophic forgetting on base capabilities

**6. Merge & Deploy**
- Merge LoRA weights back into base model
- Quantize (GPTQ/AWQ) for serving
- Deploy via vLLM, TGI, or ONNX Runtime

## Key Takeaways
- **LoRA** is the default choice — good quality with minimal compute
- Format data as instruction/response pairs matching the model's chat template
- 1-3 epochs is usually enough — more risks overfitting
- Always evaluate on held-out set + check for forgetting